# TP1a — xarray basics: open a netCDF file, select, average, and draw maps

**Prerequisites**: TP0 (you can log in to Kabre and open a notebook with the `psf2026` kernel).

[MEMO_LOCMARCA](https://docs.google.com/presentation/d/10F9j7Rs2w9Qow3FKAotAXs_TwVafaS9YMexGHde1mmg/edit?usp=sharing)

## Contents
- [0. Setup — imports and paths](#0-setup-imports-and-paths)
- [1. Open a netCDF file and look inside](#1-open-a-netcdf-file-and)
- [2. From kelvin to °C](#2-from-kelvin-to-c)
- [3. Select the part you need: `sel` and `slice`](#3-select-the-part-you-need)
- [4. A time series at one point](#exercise-2-a-time-series-at)
- [5. A sub-region and a period](#exercise-3-a-sub-region-and)
- [6. Reduce: means over a dimension](#4-reduce-means-over-a-dimension)
- [7. The mean of the box](#exercise-4-the-mean-of-the)

---
<a id="0-setup-imports-and-paths"></a>
## 0. Setup — imports and paths

Run this cell as is. It imports the libraries and defines:
- `OBS_DIR`: where the observation files are stored on Kabre (read-only, shared by everyone);
- `SST_FILE`: the file used today, **daily satellite SST** (OSTIA, 0.05°) for 2023 around Costa Rica (4–14°N, 92–77°W: Pacific **and** Caribbean sides);
- `WORK_DIR`: a folder `data_tp1/`, created next to this notebook, where **your** figures go.

| Library | What it does |
|---|---|
| `xarray` (`xr`) | reads netCDF files and handles labelled arrays (dimensions with names and coordinates) |
| `numpy` (`np`) | basic numerical arrays and maths |
| `matplotlib.pyplot` (`plt`) | figures and plots |
| `cartopy` (`ccrs`, `cfeature`) | map projections, coastlines, land, borders |

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Observation files already on Kabre (read-only)
OBS_DIR = "/data/gcambon/COMMONDATA/DATASETS_PSF/"
if not Path(OBS_DIR).exists():          # not on Kabre: instructor's local copy
    OBS_DIR = "../data/DATASETS_PSF"

SST_FILE = f"{OBS_DIR}/COSTARICA_DS/sst_costarica_2023.nc"

# Your own output files go here
WORK_DIR = Path("data_tp1")
WORK_DIR.mkdir(exist_ok=True)

print("SST file found:", Path(SST_FILE).exists())

---
<a id="1-open-a-netcdf-file-and"></a>
## 1. Open a netCDF file and look inside

`xr.open_dataset()` opens the file and returns a **Dataset**: a container holding several variables that share the same dimensions and coordinates.
It is **lazy**: at this point only the description of the file (the *header*) is read, not the data itself, so it is instant even for a big file.

Displaying the Dataset (just write its name as the last line of a cell) shows, from top to bottom:
- **Dimensions**: the names and sizes of the axes (here `time`, `latitude`, `longitude`);
- **Coordinates**: the values along each axis (the dates, the latitudes, the longitudes);
- **Data variables**: the physical fields (here `analysed_sst`), with the dimensions they depend on;
- **Attributes**: metadata about the whole file (who made it, how, from which satellites…).

Click on the small icons on the right of each line to unfold the attributes or preview the values.

In [ ]:
ds = xr.open_dataset(SST_FILE)

In [ ]:
sst_k = ds["analysed_sst"]
print("Units    :", sst_k.attrs["units"])
print("Long name:", sst_k.attrs["long_name"])
print("Time steps:", sst_k.sizes["time"])

---
<a id="2-from-kelvin-to-c"></a>
## 2. From kelvin to °C

Arithmetic on a DataArray applies to every value and keeps the dimensions and coordinates.
**But** the attributes are dropped (xarray cannot know whether `units` is still right after your operation), so we set them again ourselves.

In [ ]:
sst = sst_k - 273.15
sst.attrs = {"long_name": "SST", "units": "°C"}

> 💬 This time you see **numbers**, not just a description: `open_dataset` only read the header (section 1), but an arithmetic operation like `- 273.15` needs the values, so xarray reads the **whole variable** into memory (≈ 175 MB here, no problem).
> On a big file (a multi-year 3D model output can weigh tens of GB), **select first** (a region, a period, a depth) **and compute after**: `ds["var"].sel(...)` stays cheap, and only the selected part is read.

---
<a id="3-select-the-part-you-need"></a>
## 3. Select the part you need: `sel` and `slice`

Two ways to select, with the **name** of the dimension (never "axis 0, axis 1" as in numpy):

| Method | Selects by… | Example |
|---|---|---|
| `sel` | **value** of the coordinate | `sst.sel(time="2023-03-15")` → 15 March 2023 |
| `sel(..., method="nearest")` | closest value | `sst.sel(latitude=10.0, longitude=-87.0, method="nearest")` |
| `sel` + `slice(a, b)` | a range of values (**both ends included**) | `sst.sel(latitude=slice(8, 12))` |

A selection on one value **removes** that dimension: one day → a 2D map (lat, lon); one point → a 1D time series.

In [ ]:
day = sst.sel(time="2023-03-15")            # one day, by date
print(day.dims, "->", str(day["time"].values)[:10])

day.plot(cmap="turbo")                      # xarray quick look: one line, no map yet
plt.title("SST on 15 March 2023 (quick look)")
plt.show()

<a id="exercise-2-a-time-series-at"></a>
## 4. A time series at one point

Extract the daily SST time series at **11°N, 87°W** (offshore of the Gulf of Papagayo, north-west Costa Rica) and plot it.
Remember: west longitudes are **negative**, and the exact point is probably not on the grid → use `method="nearest"`.

In [ ]:
ts = sst.sel(latitude=11.0, longitude=-87.0, method="nearest")

print(f"Grid point used: {float(ts.latitude):.3f}°N, {float(ts.longitude):.3f}°E (negative = west)")
fig, ax = plt.subplots(figsize=(10, 3.5))
ts.plot(ax=ax)
ax.set_title("Daily SST at 11°N, 87°W — 2023")
ax.grid(alpha=0.3)
plt.show()

print(f"Min: {float(ts.min()):.2f} °C on {str(ts.idxmin().values)[:10]}")
print(f"Max: {float(ts.max()):.2f} °C on {str(ts.idxmax().values)[:10]}")


<a id="exercise-3-a-sub-region-and"></a>
## 5. A sub-region and a period

With `slice`, select the box **8–12°N, 88–84°W** for the month of **February 2023** only.
A string like `"2023-02"` selects a whole month; `slice("2023-02-01", "2023-02-28")` would do the same.

In [ ]:
box_feb = sst.sel(latitude=slice(8, 12), longitude=slice(-88, -84), time="2023-02")
print(dict(box_feb.sizes))

---
<a id="4-reduce-means-over-a-dimension"></a>
## 6. Reduce: means over a dimension

`.mean("time")` averages along `time`: 365 daily maps → **one** mean map. Same idea for `.min()`, `.max()`, `.std()`, `.sum()`… and for any dimension (`.mean(["latitude", "longitude"])` gives a time series averaged over the box).
Missing values (`NaN`, land) are **skipped** automatically.

`.groupby("time.month").mean()` puts the days into 12 groups (one per month) and averages each group: a **monthly climatology** with a new dimension `month` (1 to 12).

> ⏳ The data is already in memory since section 2, but averaging 365 maps still takes a few seconds.

In [ ]:
sst_mean = sst.mean("time")                       # annual mean map  (latitude, longitude)
sst_monthly = sst.groupby("time.month").mean()    # 12 monthly maps  (month, latitude, longitude)
print(sst_mean.dims, sst_monthly.dims)
print(f"Annual mean SST: from {float(sst_mean.min()):.1f} to {float(sst_mean.max()):.1f} °C")

<a id="exercise-4-the-mean-of-the"></a>
### 7. The mean of the box

Compute the **mean SST over the whole year** in the February box of exercise 3 (8–12°N, 88–84°W)

In [ ]:
box = sst.sel(latitude=slice(8, 12), longitude=slice(-88, -84))     # whole year this time
val_xr = float(box.mean())